# Day 10：模块化包管理与自定义异常体系

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第 10 天！我是你的主讲兼 Tech Lead 张工。昨天我们完成了多厂商大模型适配器，今天我们要解决工程落地中最关键的两件事：**用包（Package）与 `requirements.txt` 规范代码与依赖**，以及**用自定义异常体系（`ZhiShuError`）和 `try/except/else/finally` 打造可观测、不崩溃的容错链路**。

## 1. 第一步：设计智枢平台自定义异常继承树
所有业务异常统一继承自 `ZhiShuError(Exception)`，方便上层按细粒度捕获，也能用基类兜底：
- `InvalidRoleError`：消息角色不在白名单内
- `ModelConfigError`：模型名称或 `temperature` 配置非法
- `APIResponseError`：上游大模型接口超时或服务不可用

In [1]:
class ZhiShuError(Exception):
    """智枢平台统一异常基类"""
    def __init__(self, message: str, error_code: str = 'ZHISHU_ERROR') -> None:
        self.message = message
        self.error_code = error_code
        super().__init__(f'[{error_code}] {message}')

class InvalidRoleError(ZhiShuError):
    def __init__(self, role: object, valid_roles: tuple[str, ...]) -> None:
        self.role = role
        self.valid_roles = valid_roles
        super().__init__(f'无效的消息角色: {role!r}，允许的角色为 {valid_roles}', 'INVALID_ROLE')

class ModelConfigError(ZhiShuError):
    def __init__(self, message: str) -> None:
        super().__init__(message, 'MODEL_CONFIG_ERROR')

class APIResponseError(ZhiShuError):
    def __init__(self, message: str, status_code: int | None = None) -> None:
        self.status_code = status_code
        detail = f'{message} (status_code={status_code})' if status_code is not None else message
        super().__init__(detail, 'API_RESPONSE_ERROR')

print('InvalidRoleError 是 ZhiShuError 子类吗？', issubclass(InvalidRoleError, ZhiShuError))
print('APIResponseError 示例:', APIResponseError('上游服务繁忙', status_code=503))

InvalidRoleError 是 ZhiShuError 子类吗？ True
APIResponseError 示例: [API_RESPONSE_ERROR] 上游服务繁忙 (status_code=503)


## 2. 第二步：从 `project_iterations/day_10` 导入 `zhishu_agent` 包
在实际工程中，我们把模块放进 `zhishu_agent/` 目录，并在 `__init__.py` 中通过 `__all__` 统一导出。

In [2]:
import sys
from pathlib import Path

# 定位到 project_iterations/day_10 以便导入包
cwd = Path.cwd().resolve()
repo_root = next(p for p in [cwd, *cwd.parents] if (p / 'project_iterations' / 'day_10').exists())
day10_dir = repo_root / 'project_iterations' / 'day_10'
if str(day10_dir) not in sys.path:
    sys.path.insert(0, str(day10_dir))

from zhishu_agent import (
    APIResponseError,
    ChatMessage,
    InvalidRoleError,
    ModelConfigError,
    OpenAIModel,
    QwenModel,
    ZhiShuError,
)

print('成功导入 zhishu_agent 包！OpenAIModel =', OpenAIModel)

成功导入 zhishu_agent 包！OpenAIModel = <class 'zhishu_agent.models.OpenAIModel'>


## 3. 第三步：演示 `try / except / else / finally` 完整执行链路
下面我们编写 `demo_safe_call()`，分别测试：
1. 正常调用（触发 `try -> else -> finally`）
2. 传入非法角色 `'admin'`（触发 `InvalidRoleError -> finally`）
3. 传入非法温度 `3.5`（触发 `ModelConfigError`）
4. 模拟上游 API 504 超时（触发 `APIResponseError -> finally`）

In [3]:
def demo_safe_call(model, role: str, prompt: str) -> str:
    try:
        msg = ChatMessage(role=role, content=prompt)
        reply = model.generate([msg])
    except InvalidRoleError as exc:
        print(f'  [except-角色异常] {exc}')
        return 'FALLBACK_ROLE'
    except ModelConfigError as exc:
        print(f'  [except-配置异常] {exc}')
        return 'FALLBACK_CONFIG'
    except APIResponseError as exc:
        print(f'  [except-接口异常] {exc} | HTTP={exc.status_code}')
        return 'FALLBACK_API'
    except ZhiShuError as exc:
        print(f'  [except-兜底平台异常] {exc}')
        return 'FALLBACK_ZHISHU'
    else:
        print(f'  [else-成功分支] 模型返回: {reply}')
        return reply.content
    finally:
        print(f'  [finally-审计收尾] 已完成本次调用审计 (model={model.model_name})')

openai_llm = OpenAIModel(model_name='gpt-4o-mini', temperature=0.4)
qwen_llm = QwenModel(model_name='qwen-plus', temperature=0.8, enable_search=True)

print('--- 场景1: 正常调用 ---')
demo_safe_call(openai_llm, 'user', '什么是模块化包管理？')

print('\n--- 场景2: 非法角色 ---')
demo_safe_call(qwen_llm, 'admin', '你好')

print('\n--- 场景3: 非法温度配置 ---')
try:
    openai_llm.temperature = 3.5
except ModelConfigError as exc:
    print('  捕获到 ModelConfigError:', exc)

print('\n--- 场景4: 上游 API 超时 ---')
demo_safe_call(qwen_llm, 'user', '[SIMULATE_ERROR] 模拟超时')

--- 场景1: 正常调用 ---
  [else-成功分支] 模型返回: [assistant] [OpenAI:gpt-4o-mini|temp=0.4] 已收到你的消息(1轮上下文)：什么是模块化包管理？
  [finally-审计收尾] 已完成本次调用审计 (model=gpt-4o-mini)

--- 场景2: 非法角色 ---
  [except-角色异常] [INVALID_ROLE] 无效的消息角色: 'admin'，允许的角色为 ('system', 'user', 'assistant', 'tool')
  [finally-审计收尾] 已完成本次调用审计 (model=qwen-plus)

--- 场景3: 非法温度配置 ---
  捕获到 ModelConfigError: [MODEL_CONFIG_ERROR] temperature 必须在 0.0 到 2.0 之间，当前值为 3.5

--- 场景4: 上游 API 超时 ---
  [except-接口异常] [API_RESPONSE_ERROR] 通义千问 DashScope 接口响应超时 (status_code=504) | HTTP=504
  [finally-审计收尾] 已完成本次调用审计 (model=qwen-plus)


'FALLBACK_API'